In [ ]:
# =====================================================
# SCRIPT DESCRIPTION:
# =====================================================
# This script performs the following actions to prepare AIS data for a LangGraph Agent:
# 1. Connects to a Supabase (PostgreSQL) database.
# 2. Loads a local AIS CSV file into a Pandas DataFrame.
# 3. Converts the 'BaseDateTime' column into a proper SQL Timestamp format.
# 4. Converts 'LAT' and 'LON' columns into a PostGIS 'geometry' column (Point) for spatial queries.
# 5. Renames ALL columns to lowercase (e.g., 'MMSI' -> 'mmsi') to fix PostgreSQL case-sensitivity issues.
# 6. Uploads the final clean dataset to the 'ais_signals' table, replacing any existing data.

# =====================================================
# PREREQUISITES:
# - pip install pandas geopandas sqlalchemy psycopg2-binary geoalchemy2
# - Supabase project with PostGIS extension enabled.


import pandas as pd
import geopandas as gpd
from sqlalchemy import create_engine

import os
from dotenv import load_dotenv
load_dotenv()

# --- 1. SETUP CONNECTION ---
# Paste the string you copied from Supabase.

db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")
db_host = os.getenv("DB_HOST")
db_port = os.getenv("DB_PORT")
db_name = os.getenv("DB_NAME")

connection_str = f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"

# --- 2. LOAD DATA ---
csv_file = "AIS_2024_01_Combined_clean_HighFreq.csv" # <--- REPLACE WITH YOUR CSV FILENAME
print(f"Reading {csv_file}...")
df = pd.read_csv(csv_file)

# --- 3. CLEAN & PREPARE DATA ---

# A. Convert Time
# This ensures you can ask "Where was the ship at 2 PM?"
print("Parsing 'BaseDateTime' to timestamp...")
df['BaseDateTime'] = pd.to_datetime(df['BaseDateTime'])

# B. Create Geospatial Geometry
# We use the original UPPERCASE names here because the dataframe hasn't been renamed yet.
print("Converting LAT/LON to PostGIS Geometry...")
gdf = gpd.GeoDataFrame(
    df, 
    geometry=gpd.points_from_xy(df.LON, df.LAT), # Longitude is X, Latitude is Y
    crs="EPSG:4326" # Standard GPS coordinate system
)

# C. [THE FIX] Rename columns to Lowercase
# We apply this to 'gdf' (the object being uploaded), ensuring the DB gets lowercase names.
print("Standardizing column names to lowercase...")
gdf.columns = gdf.columns.str.lower()

# --- 4. UPLOAD TO SUPABASE ---
print("Connecting to database...")
engine = create_engine(connection_str)

print("Uploading to table 'ais_signals'... (This may take time)")
# if_exists="replace" will DROP the old table and create a new one with the correct lowercase columns
gdf.to_postgis("ais_signals", engine, if_exists="replace", index=False, chunksize=1000)

# --- 5. VERIFICATION ---
print("\nSuccess! Data uploaded.")
print("The table 'ais_signals' now has these columns (all lowercase):")
print(list(gdf.columns))

Reading AIS_2024_01_Combined_clean_HighFreq.csv...
Parsing 'BaseDateTime' to timestamp...
Converting LAT/LON to PostGIS Geometry...
Standardizing column names to lowercase...
Connecting to database...
Uploading to table 'ais_signals'... (This may take time)

Success! Data uploaded.
The table 'ais_signals' now has these columns (all lowercase):
['mmsi', 'basedatetime', 'lat', 'lon', 'sog', 'cog', 'heading', 'vesselname', 'imo', 'callsign', 'vesseltype', 'status', 'length', 'width', 'draft', 'cargo', 'transceiverclass', 'geometry']
